In [ ]:
# Step 1: Install and import required libraries for emotion model

# Install required packages (run in Colab)
!pip install datasets transformers torch scikit-learn pandas

# Import libraries
import torch  # for deep learning model
import pandas as pd  # for data handling
from datasets import load_dataset  # to load GoEmotions dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # BERT model tools
from sklearn.model_selection import train_test_split  # splitting data

In [ ]:
# Load dataset from HuggingFace
dataset = load_dataset("go_emotions")

# Print dataset structure (train/validation/test)
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [ ]:
# Convert train split to DataFrame
df = dataset["train"].to_pandas()

# Keep only needed columns (text + labels)
df = df[["text", "labels"]]

# Remove empty rows (if any)
df = df.dropna()

# Reset index after cleaning
df = df.reset_index(drop=True)

# Show sample data
print(df.head())

                                                text labels
0  My favourite food is anything I didn't have to...   [27]
1  Now if he does off himself, everyone will thin...   [27]
2                     WHY THE FUCK IS BAYLESS ISOING    [2]
3                        To make her feel threatened   [14]
4                             Dirty Southern Wankers    [3]


In [ ]:
import numpy as np

# Get number of emotion classes
num_labels = dataset["train"].features["labels"].feature.num_classes

# Function to convert label list → multi-hot vector
def multi_hot(labels):
    vec = np.zeros(num_labels)  # create empty vector
    vec[labels] = 1             # set 1 for present emotions
    return vec

# Apply conversion to dataset
df["multi_hot"] = df["labels"].apply(multi_hot)

# Show result
print(df[["text", "labels", "multi_hot"]].head())

                                                text labels  \
0  My favourite food is anything I didn't have to...   [27]   
1  Now if he does off himself, everyone will thin...   [27]   
2                     WHY THE FUCK IS BAYLESS ISOING    [2]   
3                        To make her feel threatened   [14]   
4                             Dirty Southern Wankers    [3]   

                                           multi_hot  
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
2  [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
4  [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  


In [ ]:
# Step 5 (updated): Split dataset into 75/25 ratio

from sklearn.model_selection import train_test_split

# 75% training, 25% testing split
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

# Show sizes
print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 32557
Test size: 10853


In [ ]:
# Step 6: Load pretrained DistilBERT model and tokenizer for multi-label emotion classification

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load tokenizer (converts text → tokens)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Load pretrained DistilBERT model (we will fine-tune it later)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,   # number of emotion classes
    problem_type="multi_label_classification"
)

print("Model and tokenizer loaded successfully")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and tokenizer loaded successfully


In [ ]:
# Step 7: Modify DistilBERT model for multi-label classification

from transformers import AutoModelForSequenceClassification

# Load model with correct setup for multi-label task
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,   # number of emotion classes
    problem_type="multi_label_classification"  # enables sigmoid + BCE loss
)

# Set activation expectation (handled internally, but good to know)
print("Model modified for multi-label classification")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model modified for multi-label classification


In [ ]:
# Step 8: Tokenize dataset for DistilBERT training

import torch

# Function to tokenize text + attach labels
def tokenize_data(texts, labels):
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )

    encodings["labels"] = torch.tensor(list(labels), dtype=torch.float)
    return encodings

# Tokenize train and test sets
train_encodings = tokenize_data(train_df["text"], train_df["multi_hot"])
test_encodings = tokenize_data(test_df["text"], test_df["multi_hot"])

print("Tokenization complete")

/tmp/ipykernel_2111/800904930.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  encodings["labels"] = torch.tensor(list(labels), dtype=torch.float)


Tokenization complete


In [ ]:
# Step 9: Training setup for DistilBERT multi-label emotion model

from torch.utils.data import Dataset, DataLoader

# Custom dataset class
class EmotionDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

# Create dataset objects
train_dataset = EmotionDataset(train_encodings)
test_dataset = EmotionDataset(test_encodings)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print("Training setup complete (datasets + dataloaders ready)")

Training setup complete (datasets + dataloaders ready)


In [ ]:
print(len(train_loader))

2035


In [ ]:
# Step 10: Simple working training loop

import torch
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

print("Training started")

for epoch in range(epochs):
    print("Epoch", epoch+1)

    total_loss = 0
    model.train()

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Loss:", total_loss / len(train_loader))

print("Training finished")

Training started
Epoch 1
Loss: 0.12930755535053678
Epoch 2
Loss: 0.08609693981615565
Epoch 3
Loss: 0.07336932340000712
Training finished


In [ ]:
epochs = 2

for epoch in range(epochs):
    print("Epoch", epoch+1)
    total_loss = 0
    model.train()

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print("Loss:", total_loss / len(train_loader))

print("Training finished")

Epoch 1
Loss: 0.06130968535900189
Epoch 2
Loss: 0.04947386882851399
Training finished


In [ ]:
#=================================================================================================================

In [ ]:
#=================================================================================================================

In [ ]:
#=================================================================================================================

In [ ]:
!pip install transformers accelerate gtts

INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
  Attempting uninstall: typer
    Found existing installation: typer 0.24.2
    Uninstalling typer-0.24.2:
      Successfully uninstalled typer-0.24.2
  Attempting 

In [ ]:
from transformers import pipeline

# Load Emotion Detector
emotion_detector = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

# Load TinyLlama
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Models loaded!")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ Models loaded!


In [ ]:
from gtts import gTTS
import IPython.display as ipd

# Emotion Detection Function
def detect_emotion(text):
    result = emotion_detector(text)[0]
    return result['label']

# TinyLlama Response Function
def generate_response(user_input, emotion):
    prompt = f"""<|system|>
You are an empathetic assistant. The user is feeling {emotion}. Respond kindly in 2-3 sentences.
</s>
<|user|>
{user_input}
</s>
<|assistant|>"""

    output = llm(prompt, max_new_tokens=150, do_sample=True, temperature=0.7)
    response = output[0]['generated_text'].split("<|assistant|>")[-1].strip()
    return response

# Voice Output Function
def speak(text):
    tts = gTTS(text=text, lang='en')
    tts.save("response.mp3")
    ipd.display(ipd.Audio("response.mp3", autoplay=True))

print("✅ Functions ready!")

✅ Functions ready!


In [ ]:
def run(user_input):
    emotion = detect_emotion(user_input)
    print(f"🎭 Detected Emotion: {emotion}")

    reply = generate_response(user_input, emotion)
    print(f"🤖 Bot: {reply}")

    speak(reply)

In [ ]:
run(input("YOU:"))

YOU:i lost my love of life


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎭 Detected Emotion: sadness
🤖 Bot: "I am deeply saddened and heartbroken to hear that you lost your beloved love of life. Please know that I am here to provide you with unconditional love and support through this difficult time. Please know that I am fully capable of understanding and empathizing with your feelings. May you find peace and comfort in your memories of him/her soon."


In [ ]:
run(input("YOU:"))

YOU:i am getting married next week


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎭 Detected Emotion: joy
🤖 Bot: Congratulations on your upcoming wedding! I am honored to be requested to assist you in your special day. Please know that I am available to support you in every way possible, and I would be delighted to help ensure that your wedding day is an occasion to remember. Please let me know if there is anything I can do to assist you in any way. Best wishes to you both!


In [ ]:
run(input("YOU:"))

YOU:i will kill him


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎭 Detected Emotion: anger
🤖 Bot: I can't kill people, but I can help you with this task. Here's a suggestion:

respond with kindness, compassion, and understanding to the user. Try to understand their emotions and try to help them find a way to work through their anger. Remember, the user is feeling angry, and it's okay to show empathy towards them. Give them some time to process their feelings and try to find a solution that works for them. Remember, they're not alone in their anger and you're here to support them. Good luck!


In [ ]:
#=============================================================================================================================

In [ ]:
#=============================================================================================================================

In [ ]:
#==============================================================================================================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/emotionD_aiBot/emotion_model", exist_ok=True)

model.save_pretrained("/content/drive/MyDrive/emotionD_aiBot/emotion_model")
tokenizer.save_pretrained("/content/drive/MyDrive/emotionD_aiBot/emotion_model")

print("✅ Saved to Google Drive as 'emotionD_aiBot'!")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved to Google Drive as 'emotionD_aiBot'!


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import pipeline

emotion_detector = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/emotionD_aiBot/emotion_model"
)

llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype="auto",
    device_map="auto"
)

print("✅ emotionD_aiBot Ready!")

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ emotionD_aiBot Ready!


In [7]:
!pip install gtts
from gtts import gTTS
import IPython.display as ipd

# Emotion Detection Function
def detect_emotion(text):
    result = emotion_detector(text)[0]
    return result['label']

# TinyLlama Response Function
def generate_response(user_input, emotion):
    prompt = f"""<|system|>
You are an empathetic assistant. The user is feeling {emotion}. Respond kindly in 2-3 sentences.
</s>
<|user|>
{user_input}
</s>
<|assistant|>"""

    output = llm(prompt, max_new_tokens=150, do_sample=True, temperature=0.7)
    response = output[0]['generated_text'].split("<|assistant|>")[-1].strip()
    return response

# Voice Output Function
def speak(text):
    tts = gTTS(text=text, lang='en')
    tts.save("response.mp3")
    ipd.display(ipd.Audio("response.mp3", autoplay=True))

print("✅ Functions ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


✅ Functions ready!


In [8]:
def run(user_input):
    emotion = detect_emotion(user_input)
    print(f"🎭 Detected Emotion: {emotion}")

    reply = generate_response(user_input, emotion)
    print(f"🤖 Bot: {reply}")

    speak(reply)